In [57]:

import os
import re
import random
import math
from collections import Counter
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader

In [ ]:
# ---------------------------
# Config
# ---------------------------
DATA_PATH = "C:\\Users\\aman\\Desktop\\Compatika-V1-alpha-ai-model-in-2-4-MB-dataset-and-1m-param\\compatika_v1alpha_dataset.txt"   # uploaded dataset path
MODEL_PATH = "compatika_lstm.pth"
EMBED_SIZE = 128
HIDDEN_SIZE = 256
NUM_LAYERS = 2
BIDIRECTIONAL = False
BATCH_SIZE = 32
EPOCHS = 30
LR = 0.001
MAX_LEN = 60      # truncate/pad sequences
TEACHER_FORCING_RATIO = 0.5
MIN_FREQ = 1      # min freq to keep in vocab (1 keeps all)
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


In [59]:
# ---------------------------
# Helpers: load & parse data
# ---------------------------
def load_pairs(path):
    """
    Parse dataset file into list of (user, compatika) pairs.
    File format expected: many lines containing "USER: ..." and "COMPATIKA: ..."
    Returns list of (input_text, target_text).
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} not found.")
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        raw = f.read()
    # Normalize line endings
    raw = raw.replace("\r\n", "\n").replace("\r", "\n")
    # A robust regex to capture pairs
    # Find all occurrences: USER: (text) \n COMPATIKA: (text)
    pattern = re.compile(r"USER:\s*(.+?)\n\s*COMPATIKA:\s*(.+?)(?:\n|$)", re.IGNORECASE | re.DOTALL)
    pairs = []
    for m in pattern.finditer(raw):
        u = m.group(1).strip()
        c = m.group(2).strip()
        # Clean short tokens if any
        if u and c:
            pairs.append((u, c))
        # If no pairs found with regex, fallback to line-by-line scanning
    if not pairs:
        lines = [l.strip() for l in raw.splitlines() if l.strip()]
        i = 0
        while i < len(lines)-1:
            if lines[i].upper().startswith("USER:") and lines[i+1].upper().startswith("COMPATIKA:"):
                u = lines[i][len("USER:"):].strip()
                c = lines[i+1][len("COMPATIKA:"):].strip()
                pairs.append((u, c))
                i += 2
            else:
                i += 1
    return pairs

pairs = load_pairs(DATA_PATH)
print("Loaded pairs:", len(pairs))
if len(pairs) == 0:
    raise RuntimeError("No USER/COMPATIKA pairs found in dataset.")        

Loaded pairs: 20000


In [60]:
# ---------------------------
# Tokenization & Vocab
# ---------------------------
def simple_tokenize(text):
    # lowercase and simple whitespace/punct split
    text = text.strip()
    # keep punctuation as separate tokens
    tokens = re.findall(r"\w+|[^\s\w]", text.lower(), re.UNICODE)
    return tokens

# Collect tokens
all_src = []
all_tgt = []
for u,c in pairs:
    all_src.extend(simple_tokenize(u)[:MAX_LEN])
    all_tgt.extend(simple_tokenize(c)[:MAX_LEN])

counter = Counter(all_src + all_tgt)

In [61]:
# Build vocab
specials = ["<pad>", "<unk>", "<sos>", "<eos>"]
vocab_tokens = [tok for tok,freq in counter.items() if freq >= MIN_FREQ]
# Sort for stability: specials + frequent tokens sorted by freq desc
vocab_tokens_sorted = sorted(vocab_tokens, key=lambda t: (-counter[t], t))
itos = specials + vocab_tokens_sorted
stoi = {tok:i for i,tok in enumerate(itos)}
PAD_IDX = stoi["<pad>"]
UNK_IDX = stoi["<unk>"]
SOS_IDX = stoi["<sos>"]
EOS_IDX = stoi["<eos>"]

print("Vocab size:", len(itos))

def encode_tokens(tokens, max_len=MAX_LEN, add_eos=True):
    ids = [stoi.get(t, UNK_IDX) for t in tokens[:max_len]]
    if add_eos:
        ids = ids + [EOS_IDX]
    return torch.tensor(ids, dtype=torch.long)

Vocab size: 146


In [62]:
# Dataset & Collate
# ---------------------------
class CompatikaDataset(Dataset):
    def __init__(self, pairs, tokenize):
        self.pairs = pairs
        self.tokenize = tokenize
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        u,c = self.pairs[idx]
        src = encode_tokens(self.tokenize(u), add_eos=True)
        tgt = encode_tokens(self.tokenize(c), add_eos=True)
        return src, tgt

def collate_fn(batch):
    srcs, tgts = zip(*batch)
    srcs_p = pad_sequence(srcs, batch_first=True, padding_value=PAD_IDX)
    tgts_p = pad_sequence(tgts, batch_first=True, padding_value=PAD_IDX)
    return srcs_p, tgts_p

In [63]:
# Shuffle and split small train/test
random.shuffle(pairs)
split = int(0.9 * len(pairs))
train_pairs = pairs[:split]
val_pairs = pairs[split:] if split < len(pairs) else pairs[:max(1, int(0.1*len(pairs)))]

train_ds = CompatikaDataset(train_pairs, simple_tokenize)
val_ds = CompatikaDataset(val_pairs, simple_tokenize)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

In [64]:
# Model: Encoder & Decoder (LSTM)
# ---------------------------
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_size, hid_size, n_layers=1, bidir=False):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_size, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(emb_size, hid_size, num_layers=n_layers, batch_first=True, bidirectional=bidir)
        self.bidir = bidir
        self.n_layers = n_layers
        self.hid_size = hid_size
    def forward(self, x, lengths=None):
        # x: (B, T)
        e = self.emb(x)  # (B, T, E)
        outputs, (h, c) = self.lstm(e)  # outputs (B, T, H*dir)
        # If bidir, we may want to combine states; for simplicity, we'll keep as-is.
        return outputs, (h, c)

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_size, hid_size, n_layers=1):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_size, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(emb_size, hid_size, num_layers=n_layers, batch_first=True)
        self.fc = nn.Linear(hid_size, vocab_size)
    def forward(self, input_tokens, hidden):
        # input_tokens: (B, 1) single step
        e = self.emb(input_tokens)  # (B,1,E)
        output, hidden = self.lstm(e, hidden)  # output (B,1,H)
        logits = self.fc(output.squeeze(1))  # (B, V)
        return logits, hidden

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
    def forward(self, src, tgt=None, teacher_forcing_ratio=0.5, max_len=MAX_LEN+1):
        batch_size = src.size(0)
        vocab_size = self.decoder.fc.out_features
        # Encode
        enc_out, hidden = self.encoder(src)
        # Prepare decoder inputs
        dec_input = torch.full((batch_size,1), SOS_IDX, dtype=torch.long, device=self.device)
        dec_hidden = hidden  # use encoder hidden as initial (works reasonably for same hidden sizes)
        outputs = torch.zeros(batch_size, max_len, vocab_size, device=self.device)
        for t in range(0, max_len):
            logits, dec_hidden = self.decoder(dec_input, dec_hidden)
            outputs[:, t, :] = logits
            teacher_force = (t < (tgt.size(1) if tgt is not None else 0)) and (random.random() < teacher_forcing_ratio)
            if teacher_force and tgt is not None:
                dec_input = tgt[:, t].unsqueeze(1).to(self.device)  # use actual token
            else:
                top1 = logits.argmax(dim=1).unsqueeze(1)
                dec_input = top1
        return outputs

In [65]:
# Instantiate
vocab_size = len(itos)
enc = Encoder(vocab_size, EMBED_SIZE, HIDDEN_SIZE, n_layers=NUM_LAYERS, bidir=BIDIRECTIONAL)
dec = Decoder(vocab_size, EMBED_SIZE, HIDDEN_SIZE, n_layers=NUM_LAYERS)
model = Seq2Seq(enc, dec, device).to(device)
print("Model created. Parameters:", sum(p.numel() for p in model.parameters()))

# Loss and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

Model created. Parameters: 1918098


In [66]:
# Training loop
# ---------------------------
def train_epoch(model, loader, optimizer, criterion, teacher_forcing_ratio):
    model.train()
    total_loss = 0.0
    for src, tgt in loader:
        src = src.to(device)
        tgt = tgt.to(device)
        optimizer.zero_grad()
        out = model(src, tgt=tgt, teacher_forcing_ratio=teacher_forcing_ratio, max_len=tgt.size(1))
        # out: (B, T, V)  -> compute loss against tgt
        out_flat = out.view(-1, out.size(-1))
        tgt_flat = tgt.view(-1)
        loss = criterion(out_flat, tgt_flat)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * src.size(0)
    return total_loss / len(loader.dataset)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for src, tgt in loader:
            src = src.to(device)
            tgt = tgt.to(device)
            out = model(src, tgt=tgt, teacher_forcing_ratio=0.0, max_len=tgt.size(1))
            out_flat = out.view(-1, out.size(-1))
            tgt_flat = tgt.view(-1)
            loss = criterion(out_flat, tgt_flat)
            total_loss += loss.item() * src.size(0)
    return total_loss / len(loader.dataset)

best_val = float("inf")
for epoch in range(1, EPOCHS+1):
    tr_loss = train_epoch(model, train_loader, optimizer, criterion, TEACHER_FORCING_RATIO)
    val_loss = evaluate(model, val_loader, criterion)
    print(f"Epoch {epoch:02d} | train_loss={tr_loss:.4f} | val_loss={val_loss:.4f}")
    # simple save
    if val_loss < best_val:
        best_val = val_loss
        torch.save({
            "model_state": model.state_dict(),
            "itos": itos,
            "stoi": stoi,
            "config": {
                "embed": EMBED_SIZE, "hidden": HIDDEN_SIZE, "layers": NUM_LAYERS
            }
        }, MODEL_PATH)
        print("  -> saved best model")

Epoch 01 | train_loss=1.3123 | val_loss=3.1018
  -> saved best model
Epoch 02 | train_loss=0.3162 | val_loss=3.5720
Epoch 03 | train_loss=0.2926 | val_loss=3.4240
Epoch 04 | train_loss=0.2962 | val_loss=3.5355
Epoch 05 | train_loss=0.2937 | val_loss=3.4019
Epoch 06 | train_loss=0.2669 | val_loss=3.3084
Epoch 07 | train_loss=0.2794 | val_loss=3.3708
Epoch 08 | train_loss=0.2608 | val_loss=3.8256
Epoch 09 | train_loss=0.2759 | val_loss=3.4944
Epoch 10 | train_loss=0.2607 | val_loss=3.6316
Epoch 11 | train_loss=0.2851 | val_loss=3.2078
Epoch 12 | train_loss=0.2691 | val_loss=3.3458
Epoch 13 | train_loss=0.2652 | val_loss=3.5464
Epoch 14 | train_loss=0.2776 | val_loss=3.6289
Epoch 15 | train_loss=0.2760 | val_loss=3.0626
  -> saved best model
Epoch 16 | train_loss=0.2516 | val_loss=3.3938
Epoch 17 | train_loss=0.2599 | val_loss=3.3735
Epoch 18 | train_loss=0.2582 | val_loss=3.4782
Epoch 19 | train_loss=0.2517 | val_loss=3.2524
Epoch 20 | train_loss=0.2452 | val_loss=3.3291
Epoch 21 | train

In [69]:
# ---------------------------
# Inference helpers
# ---------------------------
def decode_ids(ids):
    # convert list of token ids to string, stop at <eos>
    toks = []
    for i in ids:
        if i == EOS_IDX:
            break
        if i == PAD_IDX or i == SOS_IDX:
            continue
        toks.append(itos[i])
    return " ".join(toks)

def predict(model, raw_text, max_len=MAX_LEN+1):
    model.eval()
    tokens = simple_tokenize(raw_text)
    src = encode_tokens(tokens, add_eos=True).unsqueeze(0).to(device)  # (1, T)
    with torch.no_grad():
        out = model(src, tgt=None, teacher_forcing_ratio=0.0, max_len=max_len)  # (1, T, V)
    ids = out.argmax(dim=2).squeeze(0).tolist()
    return decode_ids(ids)


In [70]:
# ---------------------------
# Show examples (train/dev) and test predictions
# ---------------------------
print("\n--- Example dataset pairs (a few) ---")
for i in range(5):
    u,c = pairs[i]
    print(f"USER: {u}")
    print(f"COMPATIKA (gold): {c}")
    print("---")

print("\n--- Predictions on some validation examples ---")
for i in range(min(10, len(val_pairs))):
    u, gold = val_pairs[i]
    pred = predict(model, u)
    print("USER:", u)
    print("PRED:", pred)
    print("GOLD:", gold)
    print("---")

print(f"\nModel saved to {MODEL_PATH}")


--- Example dataset pairs (a few) ---
USER: I became sad when I thinking about my past.
COMPATIKA (gold): That sounds heavy… I’m glad you shared it with me.
---
USER: I became angry after remembering something.
COMPATIKA (gold): I get why that made you upset — it was a lot to handle.
---
USER: Yesterday I was anxious while resting at home.
COMPATIKA (gold): It makes sense that your thoughts felt overwhelming.
---
USER: Yesterday I was grateful while going through a stressful moment.
COMPATIKA (gold): I can see why that touched you deeply.
---
USER: Yesterday I was grateful while going through a stressful moment.
COMPATIKA (gold): It sounds like something truly meaningful happened.
---

--- Predictions on some validation examples ---
USER: I was feeling proud when talking to my friend happened.
PRED: it makes sense you feel proud — that was a big moment .
GOLD: That accomplishment says a lot about you.
---
USER: It made me feel sad while talking to my friend.
PRED: it makes sense you f